# calculate&步距.ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import os
import math
from decimal import Decimal
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from google.colab import files
import io


In [ ]:
uploaded = files.upload()

all_data = pd.DataFrame()


In [ ]:
def step_x(x1, x2):
    return abs(x2 - x1)


In [ ]:
for file_name in uploaded.keys():
    if file_name.endswith(".csv"):

        df = pd.read_csv(io.BytesIO(uploaded[file_name]))


        if 'kpts' not in df.columns:
            print(f"Error: 文件 {file_name} 中沒有 'kpts' 欄位，跳過該文件。")
            continue


        kpts_columns = df['kpts'].str.split(',', expand=True)


        for i in range(kpts_columns.shape[1]):
            kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)


        df = df.join(kpts_columns)


        df.reset_index(drop=True, inplace=True)


        if df.columns.duplicated().any():
            print(f"文件 {file_name} 存在重複列，正在刪除重複列...")
            df = df.loc[:, ~df.columns.duplicated()]


        df.drop_duplicates(inplace=True)


        if 'kpt_46' in df.columns and 'kpt_49' in df.columns:
            x16 = pd.to_numeric(df['kpt_46'], errors='coerce')
            x17 = pd.to_numeric(df['kpt_49'], errors='coerce')

            step_lengths = [step_x(x16.iloc[i], x17.iloc[i]) for i in range(len(x16))]


            average_step_length = sum(step_lengths) / len(step_lengths) if step_lengths else 0
            print(f"{file_name} 的平均步距：{average_step_length:.2f}")

            df['step_length'] = step_lengths


            if all_data.empty:
                all_data = df.copy()
            else:

                common_columns = all_data.columns.intersection(df.columns)
                all_data = pd.concat([all_data[common_columns], df[common_columns]], ignore_index=True)
        else:
            print(f"文件 {file_name} 缺少必要的列 'kpt_46' 或 'kpt_49'，跳過該文件。")


In [ ]:
all_data = all_data.join(kpts_columns, rsuffix='_kpts')


In [ ]:
all_data.head(5)


In [ ]:
print(all_data)


In [ ]:
if 'kpts' not in all_data.columns:
    print("Error: 'kpts' column not found in the uploaded files.")
else:

    kpts_columns = all_data['kpts'].str.split(',', expand=True)

    for i in range(kpts_columns.shape[1]):
        kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)

    all_data = all_data.join(kpts_columns, rsuffix='_kpts')


    print(all_data.columns)


    def step_x(x1, x2):
        step_ans = abs(x2 - x1)
        return step_ans


    if 'kpt_46' not in all_data.columns or 'kpt_49' not in all_data.columns:
        print("Error: 'kpt_46' or 'kpt_49' column not found.")
    else:

        x16 = pd.to_numeric(all_data['kpt_46'], errors='coerce')
        x17 = pd.to_numeric(all_data['kpt_49'], errors='coerce')


        print(f"Missing values in 'kpt_46': {x16.isnull().sum()}")
        print(f"Missing values in 'kpt_49': {x17.isnull().sum()}")


計算步距

In [ ]:
def step_x(x1, x2):
    step_ans = abs(x2 - x1)
    return step_ans


In [ ]:
x16 = all_data['kpt_46'].astype(float)
x17 = all_data['kpt_49'].astype(float)


In [ ]:
step_lengths = []
total_step_length = 0

for i in range(len(x16)):
    step_length = step_x(x16[i], x17[i])
    step_lengths.append(step_length)
    total_step_length += step_length


all_data['step_length'] = step_lengths


In [ ]:
average_step_length = sum(step_lengths) / len(step_lengths) if step_lengths else 0


In [ ]:
all_data.to_csv('kpts_exploded.csv', index=False)


In [ ]:
print(f"總步距: {total_step_length}")
print(f"平均步距: {average_step_length}")


下載data

In [ ]:
all_data.to_csv('步距.csv', index=False)
files.download('步距.csv')
